# PPO + Hyperparameter Optimization

PPO (Schulman et al., 2017) on classic-control environments, plus a hyperparameter study
(OFAT sweeps, random search, Optuna TPE).

**Colab usage:** upload the `PPO_submission` folder to Google Drive, then set `PROJECT_DIR`
in the setup cell below to point at it.

In [ ]:
import os, sys

# --- Colab: mount Drive and point at the uploaded folder (edit this path) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/PPO_submission'
except ModuleNotFoundError:
    # Running locally: use the folder containing this notebook
    PROJECT_DIR = os.getcwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print('Working dir:', PROJECT_DIR)

# Install dependencies (Colab already has torch/numpy/matplotlib)
!pip install -q "gymnasium[classic-control]" optuna pandas

## 1. Train PPO baseline

Train one environment over multiple seeds. Saves learning curves and `summary.json` under `results/`.

In [ ]:
from src.run import train_env

out = train_env('CartPole-v1', seeds=[0, 1, 2], results_dir='results', show_plots=True)
print('Final:', out['summary']['final_eval_mean'], '+/-', out['summary']['final_eval_std_across_seeds'])

## 2. Paper baseline (all environments)

Runs paper-default PPO on every environment with extended metrics and writes `benchmark_table.json`.

In [ ]:
from src.paper_run import run_paper_ppo_all_envs

res = run_paper_ppo_all_envs(seeds=[0, 1, 2], results_dir='results/paper_baseline')
for row in res['benchmark']['rows']:
    print(f"{row['env_id']:>28}: {row['final_eval_mean']:.1f} +/- {row['final_eval_std']:.1f}")

## 3. One-factor-at-a-time (OFAT) sweep

Vary a single hyperparameter while holding the rest at the paper default, then plot the
learning curve per value and rank knobs by sensitivity.

In [ ]:
import hpo

ENV = 'CartPole-v1'

# Sweep one knob (learning rate) over its grid, 3 seeds each
sweep = hpo.sweep_one_hparam(ENV, 'lr', seeds=[0, 1, 2], total_timesteps=100_000,
                             results_dir='results/ofat')
hpo.plots.plot_ofat_curves(sweep)

# Rank several knobs by how much they move the score
rep = hpo.sweep_sensitivity_report(ENV, hparams=['lr', 'clip_eps', 'ent_coef'],
                                   seeds=[0, 1, 2], total_timesteps=100_000,
                                   results_dir='results/ofat')
hpo.plots.plot_sensitivity_bars(rep)

## 4. Joint search: Random vs Optuna TPE

Tune several hyperparameters jointly at an equal trial budget and compare best-so-far curves.
Use `hpo.JOINT_SPACE` for easy envs and `hpo.JOINT_SPACE_HARD` for hard ones.

In [ ]:
ENV = 'CartPole-v1'
N_TRIALS = 15
SEEDS = [0]   # use [0,1,2] for the final report; single seed is fine while exploring

rand = hpo.random_search(ENV, n_trials=N_TRIALS, seeds=SEEDS, total_timesteps=100_000,
                         space=hpo.JOINT_SPACE, results_dir='results/random', verbose=False)
opt = hpo.optuna_search(ENV, n_trials=N_TRIALS, seeds=SEEDS, total_timesteps=100_000,
                        space=hpo.JOINT_SPACE, results_dir='results/optuna', verbose=False)

hpo.plots.plot_search_progress({'Random': rand, 'Optuna TPE': opt})
print('Random best:', rand['best_score'], rand['best_hparams'])
print('Optuna best:', opt['best_value'], opt['best_params'])

In [ ]:
# Optuna optimization history + hyperparameter importances
hpo.plots.plot_optuna_history(opt['study'])
hpo.plots.plot_optuna_importances(opt['study'])